# Facial Keypoint Detection

Welcome back for exercise 9! As we told you, the exercises of this lecture can be subdivided into mainly two parts. The first part in which we re-invented the wheel and implemented the most important methods on our own and the second part, where we start using existing libraries (that already have implemented all the methods). It's now time to start playing around with more complex network architectures. 

We've already entered stage two, but with the introduction of Convolutional Neural Networks this week, we are given a very powerful tool that we want to explore in this exercises. Therefore, in this week's exercise your task is to build a Convolutional Neural Network to perform facial keypoint detection. 

Before we start, let's take a look at some example images and corresponding facial keypoints:

[Figure omitted from code-only export.]

The facial keypoints (also called facial landmarks) are the small magenta dots shown on each of the faces in the images above. These keypoints mark important areas of the face: the eyes, corners of the mouth, the nose, etc. and are relevant for a variety of computer vision tasks, such as face filters, emotion recognition, pose recognition, and more. 

## (Optional) Mount folder in Colab

Uncomment thefollowing cell to mount your gdrive if you are using the notebook in google colab:

In [ ]:
# Use the following lines if you want to use Google Colab
# We presume you created a folder "i2dl" within your main drive folder, and put the exercise there.
# NOTE: terminate all other colab sessions that use GPU!
# NOTE 2: Make sure the correct exercise folder (e.g exercise_09) is given.

"""
from google.colab import drive
import os

gdrive_path='/content/gdrive/MyDrive/i2dl/exercise_09'

# This will mount your google drive under 'MyDrive'
drive.mount('/content/gdrive', force_remount=True)

# In order to access the files in this notebook we have to navigate to the correct folder
os.chdir(gdrive_path)

# Check manually if all files are present
print(sorted(os.listdir()))
"""


## Set up PyTorch environment in colab
- (OPTIONAL) Enable GPU via Runtime --> Change runtime type --> GPU
- Uncomment the following cell if you are using the notebook in google colab:

In [ ]:
############ COLAB ############
# You may simply used the preinstalled versions of colab. No need to reinstall pytorch.

# 1. Preparation

## Imports

In [ ]:
import os
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from exercise_code.data.facial_keypoints_dataset import FacialKeypointsDataset
from exercise_code.networks.keypoint_nn import (
    DummyKeypointModel,
    KeypointModel
)
from exercise_code.util import (
    show_all_keypoints,
    save_model,
)
from exercise_code.tests import test_keypoint_nn
# [OLD]
# from exercise_code.data.augmentations import (
#     JointRandomHorizontalFlip, JointRandomRotation,
#     JointColorJitter, JointCompose
# )
from exercise_code.data.augmentations import (
    JointRandomHorizontalFlip, JointRandomRotation,
    JointColorJitter, JointRandomScale, JointGaussianNoise, JointCompose
)


%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# for auto-reloading external modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
import sys
import importlib
import types

# Python 3.12 removed the `imp` module; inject a shim so %autoreload 2 works on all versions
if 'imp' not in sys.modules:
    _fake_imp = types.ModuleType('imp')
    _fake_imp.reload = importlib.reload
    sys.modules['imp'] = _fake_imp

%load_ext autoreload
%autoreload 2

os.environ['KMP_DUPLICATE_LIB_OK']='True' # To prevent the kernel from dying.


<div class="alert alert-warning">
    <h3>Note: Google Colab</h3>
    <p>
In case you don't have a GPU, you can run this notebook on Google Colab where you can access a GPU for free, but you can also run this notebook on your CPU.
         </p>
</div>

In [ ]:
# This will set the device on which to run the code, which defaults to the CPU.
# If you have an Nvidia GPU, it will use that with cuda
# For Apple Silicon, you can try to use the "mps" device by commenting in line 8.
#  But it is not guaranteed to work and sometimes the CPU performs better.

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else 
    "mps" if torch.backends.mps.is_available() else 
    "cpu"
)
print('You are using the following device: ', device)

## Load and Visualize Data
To load the data, we have already prepared a Pytorch Dataset class `FacialKeypointsDataset` for you. You can find it in `exercise_code/data/facial_keypoints_dataset.py`. Run the following cell to download the data and initialize your dataset:

In [ ]:
download_url = "https://vision.in.tum.de/webshare/g/i2dl/facial_keypoints.zip"
i2dl_exercises_path = os.path.dirname(os.path.abspath(os.getcwd()))
data_root = os.path.join(i2dl_exercises_path, "datasets", "facial_keypoints")

# [OLD: with noise/scale experiments]
# train_joint_transform = JointCompose([...with noise/scale...])
train_joint_transform = JointCompose([
    JointRandomHorizontalFlip(p=0.5),
    JointColorJitter(brightness=0.3, contrast=0.3, p=0.8),
    JointRandomRotation(max_angle=10, p=0.5),
])

train_dataset = FacialKeypointsDataset(
    train=True,
    transform=transforms.ToTensor(),
    root=data_root,
    download_url=download_url,
    joint_transform=train_joint_transform,
)

val_dataset = FacialKeypointsDataset(
    train=False,
    transform=transforms.ToTensor(),
    root=data_root,
)

print("Number of training samples:", len(train_dataset))
print("Number of validation samples:", len(val_dataset))


In [ ]:
print(train_dataset.key_pts_frame.columns.tolist())


Each sample in our dataset is a dict `{"image": image, "keypoints": keypoints}`, where
 * `image` is a [0-1]-normalized gray-scale image of size 96x96, represented by a torch tensor of shape (CxHxW) with C=1, H=96, W=96
    [Figure omitted from code-only export.]
 * `keypoints` is the list of K facial keypoints, stored in a torch tensor of shape (Kx2). We have K=15 keypoints that stand for:
   * keypoints[0]: Center of the left eye
   * keypoints[1]: Center of the right eye
   * keypoints[2]: Left eye inner corner
   * keypoints[3]: Left eye outer corner
   * keypoints[4]: Right eye inner corner
   * keypoints[5]: Right eye outer corner
   * keypoints[6]: Left eyebrow inner end
   * keypoints[7]: Left eyebrow outer end
   * keypoints[8]: Right eyebrow inner end
   * keypoints[9]: Right eyebrow outer end
   * keypoints[10]: Nose tip
   * keypoints[11]: Mouth left corner
   * keypoints[12]: Mouth right corner
   * keypoints[13]: Mouth center top lip
   * keypoints[14]: Mouth center bottom lip
   
Each individual facial keypoint is represented by two coordinates (x,y) that specify the horizontal and vertical location of the keypoint respectively. All keypoint values are normalized to be in the range [-1,1], such that:
   * (x=-1,y=-1) corresponds to the top left corner, 
   * (x=-1,y=1) to the bottom left corner,
   * (x=1,y=-1) to the top right corner,
   * (x=1,y=1) to the bottom right corner,
   * and (x=0,y=0) to the center of the image.
   
      
The data downloaded is already preprocessed and hence there is no need to apply transformations in order to prepare the data. Of course, feel free to apply training transformations to improve your performance such as e.g. flipping the training images. </br>

**Note**: The data downloaded is already preprocessed and hence there is **no need** to apply normalization transformations in order to prepare the data. Of course, feel free to apply training transformations to improve your performance such as e.g. flipping the training images. </br>

Also, when applying transformations such as flipping, make sure that the predicted coordinates of your keypoints change accordingly.

Let's have a look at the first training sample to get a better feeling for the dataset:

In [ ]:
image, keypoints = train_dataset[0]["image"], train_dataset[0]["keypoints"]
print("Shape of the image:", image.size())
print("Smallest value in the image:", torch.min(image))
print("Largest value in the image:", torch.max(image))
print(image)


In [ ]:
print(keypoints)


In `exercise_code/util/vis_utils.py` we also provide you with a function `show_all_keypoints()` that takes in an image and keypoints and displays where the predicted keypoints are in the image. Let's use it to plot the first few images of our training set:

**Note:** if your kernel dies when running the following cell, please uncomment the last line of the imports cell `os.environ['KMP_DUPLICATE_LIB_OK']='True'`and try it again. 

In [ ]:
def show_keypoints(dataset, num_samples=3):
    for i in range(num_samples):
        image = dataset[i]["image"]
        key_pts = dataset[i]["keypoints"]
        show_all_keypoints(image, key_pts)


show_keypoints(train_dataset)


# 2. Facial Keypoint Detection
Your task is to define and train a model for facial keypoint detection.

The facial keypoint detection task can be seen as a regression problem, where the goal is to predict 30 different values that correspond to the 15 facial keypoint locations. Thus, we need to build a network that gets a (1x96x96) image as input and predicts 30 continuous outputs between [-1,1].

## Dummy Model

<div class="alert alert-success">
    <h3>Task: Check Code</h3>
    <p>In <code>exercise_code/networks/keypoint_nn.py</code> we defined a naive <code>DummyKeypointModel</code>, which always predicts the keypoints of the first training image in the dataset. Let's try it on a few images and visualize our predictions in red:
 </p>
</div>

In [ ]:
def show_keypoint_predictions(model, dataset, num_samples=3):
    for i in range(num_samples):
        image = dataset[i]["image"].to(device)
        key_pts = dataset[i]["keypoints"].to(device)
        predicted_keypoints = torch.squeeze(model(image).detach()).view(15, 2)
        show_all_keypoints(image, key_pts, predicted_keypoints)


In [ ]:
dummy_model = DummyKeypointModel()
show_keypoint_predictions(dummy_model, train_dataset)


As we see, the model predicts the first sample perfectly, but for the remaining samples the predictions are quite off.

## Loss and Metrics

To measure the quality of the model's predictions, we will use the mean squared error (https://en.wikipedia.org/wiki/Mean_squared_error), summed up over all 30 keypoint locations. In PyTorch, the mean squared error is defined in `torch.nn.MSELoss()`, and we can use it like this:

In [ ]:
loss_fn = torch.nn.MSELoss()
for i in range(3):
    image = train_dataset[i]["image"]
    keypoints = train_dataset[i]["keypoints"]
    predicted_keypoints = torch.squeeze(dummy_model(image)).view(15, 2)
    loss = loss_fn(keypoints, predicted_keypoints)
    print("Loss on image %d:" % i, loss)


As expected, our dummy model achieves a loss close to 0 on the first sample, but on all other samples the loss is quite high.

To obtain an evaluation score (in the notebook and on the submission server), we will use the following function:

In [ ]:
import math
import torchvision.transforms.functional as TF_vis

# [OLD: flip-only TTA]
# def evaluate_model(model, dataset):
#     ...pred1 (original) + pred2 (flip) only...

_TTA_FLIP_PAIRS = [(0,1),(2,4),(3,5),(6,8),(7,9),(11,12)]

def _tta_rotate(model, image, angle_deg, device):
    """Predict on image rotated by angle_deg CCW, then inverse-rotate coords back."""
    angle_rad = math.radians(angle_deg)
    cos_a, sin_a = math.cos(angle_rad), math.sin(angle_rad)
    img_rot = TF_vis.rotate(image, angle_deg)
    pred = model(img_rot).view(-1, 15, 2).to(device).clone()
    x, y = pred[:, :, 0].clone(), pred[:, :, 1].clone()
    pred[:, :, 0] = x * cos_a - y * sin_a   # inverse rotation
    pred[:, :, 1] = x * sin_a + y * cos_a
    return pred

def evaluate_model(model, dataset):
    model.eval()
    model.to(device)
    criterion = torch.nn.MSELoss()
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
    loss = 0
    for batch in dataloader:
        image, keypoints = batch["image"].to(device), batch["keypoints"].to(device)
        # 1. Original
        pred1 = model(image).view(-1, 15, 2).to(device)
        # 2. Horizontal flip
        pred2 = model(image.flip(-1)).view(-1, 15, 2).to(device).clone()
        pred2[:, :, 0] *= -1
        for i, j in _TTA_FLIP_PAIRS:
            pred2[:, [i, j]] = pred2[:, [j, i]]
        # 3. Rotate +5°
        pred3 = _tta_rotate(model, image, 5, device)
        # 4. Rotate -5°
        pred4 = _tta_rotate(model, image, -5, device)
        predicted_keypoints = (pred1 + pred2 + pred3 + pred4) / 4
        loss += criterion(
            torch.squeeze(keypoints), torch.squeeze(predicted_keypoints)
        ).item()
    return 1.0 / (2 * (loss / len(dataloader)))

print("Score of the Dummy Model:", evaluate_model(dummy_model, val_dataset))


**To pass the assignment, you will need to achieve a score of at least 100**. As you can see, the score is calculated from the average loss, so **your average loss needs to be lower than 0.005**. Our dummy model only gets a score of around 60, so you will have to come up with a better model.

## Step 1: Design your own model

<div class="alert alert-info">
    <h3>Task: Implement</h3>
    <p> Now it is your turn to build your own model. To do so, you need to design a convolution neural network that takes images of size (Nx1x96x96) as input and produces outputs of shape (Nx30) in the range [-1,1]. Therefore, implement the <code>KeypointModel</code> class in <code>exercise_code/networks/keypoint_nn.py</code>.
    </p>
</div>

Recall that CNN's are defined by a few types of layers:
* Convolutional layers
* Max-pooling layers
* Fully-connected layers

You can design your network however you want, but we strongly suggest to include multiple convolution layers. You are also encouraged to use things like dropout and batch normalization to stabilize and regularize your network. If you want to build a really competitive model, have a look at some literature on keypoint detection, such as [this paper](https://arxiv.org/pdf/1710.00977.pdf).

#### Define your model in the provided file 
`exercise_code/networks/keypoint_nn.py` file

This file is mostly empty but contains the expected class name, and the methods that your model needs to implement (only `forward()` basically).

The only rules your model design has to follow are:
* Perform the forward pass in forward(), predicting keypoints of shape (Nx30) for images of shape (Nx1x96x96)
* Have less than 5 million parameters
* Have a model size of less than 20MB after saving

Furthermore, you need to pass all your hyperparameters to the model in a single dictionary:

In [ ]:
hparams = {
    "conv_channels": [32, 64, 256, 256],
    "dropout": 0.15,
    "fc_dim": 512,
    "pool_size": 6,
}

To test whether your model follows the basic rules, run the following cell:

In [ ]:
model = KeypointModel(hparams)
test_keypoint_nn(model)


## Step 2: Train your model

<div class="alert alert-info">
    <h3>Task: Implement</h3>
    <p> In addition to the network itself, you will also need to write the code for the model training.
    </p>
</div>

**Hints:**
* Use `torch.nn.MSELoss()` as loss function.

* You have two options for training code:
    - Use a straightforward training scheme. See 1_pytorch.ipynb from ex07.
    - Don't call your model anything else besides "model", unless you notice that you'll need to modify the model name in the upcoming cells.
    - You could refer to the training scheme from exercise_08 to complete your code. Understanding this pipeline is crucial for future works in deep learning.
    

In [ ]:
########################################################################
# TODO - Train Your Model                                              #
########################################################################
import math as _math
from torch.utils.data import DataLoader

def wing_loss(pred, target, w=0.04, eps=0.01):
    """Wing loss for facial landmark detection (coords in normalized [-1,1] space).
    w=0.04 ≈ 2px, eps=0.01 ≈ 0.5px at 96px image scale."""
    diff = (pred - target).abs()
    C = w - w * _math.log(1 + w / eps)
    return torch.where(diff < w,
                       w * torch.log(1 + diff / eps),
                       diff - C).mean()

batch_size = 64
epochs = 200  # 150 → 200: val_loss still decreasing at epoch 150
learning_rate = 1e-3

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)

model = KeypointModel(hparams).to(device)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=8, min_lr=1e-5
)

train_losses, val_losses, scores = [], [], []
best_val_loss    = float('inf')
best_model_state = None

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        images    = batch["image"].to(device)
        keypoints = batch["keypoints"].view(images.size(0), -1).to(device)
        optimizer.zero_grad()
        predictions = model(images)
        # [OLD: MSELoss / heatmap loss]
        # loss = criterion(predictions, keypoints)
        loss = wing_loss(predictions, keypoints)
        loss.backward()
        optimizer.step()
        train_loss += criterion(predictions, keypoints).item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            images    = batch["image"].to(device)
            keypoints = batch["keypoints"].view(images.size(0), -1).to(device)
            val_loss += criterion(model(images), keypoints).item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss   = val_loss   / len(val_loader)
    score          = 1.0 / (2 * avg_val_loss)
    scheduler.step(avg_val_loss)
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    scores.append(score)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:02d}/{epochs} | train_loss={avg_train_loss:.5f} | "
          f"val_loss={avg_val_loss:.5f} | score={score:.2f} | lr={current_lr:.2e}")

    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_model_state)
model.to(device)
print(f"\nBest val_loss={best_val_loss:.5f} | Best score={1/(2*best_val_loss):.2f}")

epochs_range = range(1, epochs + 1)
plt.figure(figsize=(10, 4))
plt.plot(epochs_range, train_losses, label="train loss")
plt.plot(epochs_range, val_losses,   label="validation loss")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss")
plt.title("Train and Validation Loss"); plt.legend(); plt.grid(True); plt.show()

plt.figure(figsize=(10, 4))
plt.plot(epochs_range, scores, label="validation score", color="green")
plt.xlabel("Epoch"); plt.ylabel("Score")
plt.title("Validation Score"); plt.legend(); plt.grid(True); plt.show()

########################################################################
#                           END OF YOUR CODE                           #
########################################################################


When you're done training, run the cells below to visualize some predictions of your model, and to compute a validation score:

In [ ]:
show_keypoint_predictions(model, val_dataset)

In [ ]:
print("Score:", evaluate_model(model, val_dataset))


# 3. Save Your Model for Submission

If your model achieved a validation score of 100 or higher, save your model with the cell below and submit it to [the submission server](https://i2dl.vc.in.tum.de). Your validation set is of course different from the test set on our server, so results may vary. Nevertheless, you will have a reasonable close approximation about your performance.

Before that, we will check again whether the number of parameters is below 5 Mio. and the file size is below 20 MB.

In [ ]:
save_model(model, "facial_keypoints.p")


Congrats - you've now finished your first Convolution Neural Network! Simply run the following cell to create a zipped file for your implementations.

In [ ]:
# Now zip the folder for upload
from exercise_code.util.submit import submit_exercise

submit_exercise("../output/exercise09")


# Submission Goals

- Goal: Implement and train a convolution neural network for facial keypoint detection.
- Passing Criteria: Reach **Score >= 100** on __our__ test dataset. The submission system will show you your score after you submit.
- You can make **$\infty$** submissions until the deadline. Your __best submission__ will be considered for bonus